# 📙 Bài 3 — Trực quan hoá

**Tuần 7**

> **Vì sao phải vẽ?** Vì con số nói dối. Bài này sẽ chứng minh điều đó bằng một ví dụ khiến bạn không bao giờ dám chỉ nhìn thống kê nữa.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25          # lưới phải MỜ, không được cạnh tranh với dữ liệu
plt.rcParams["axes.spines.top"] = False    # bỏ viền thừa
plt.rcParams["axes.spines.right"] = False

DATA = Path("../../../data")
if not (DATA / "ban_hang.csv").exists():
    DATA = Path("data")

df = pd.read_csv(DATA / "ban_hang.csv")
df["so_luong"] = pd.to_numeric(df["so_luong"], errors="coerce")
df["don_gia"] = pd.to_numeric(
    df["don_gia"].astype(str).str.replace(".", "", regex=False), errors="coerce"
)
df["thanh_tien"] = df["so_luong"] * df["don_gia"] * (1 - df["giam_gia"])
df = df[df["thanh_tien"] > 0]

print(f"Đã nạp {len(df)} đơn hàng hợp lệ")

## 0. Bốn quy tắc vẽ biểu đồ

Trước khi vẽ dòng nào, nhớ bốn điều — chúng đúng với **mọi** công cụ, mọi ngôn ngữ:

| Quy tắc | Vì sao |
|---|---|
| **Chọn dạng theo *việc* dữ liệu cần làm** | Phân bố → histogram. So sánh nhóm → bar. Quan hệ → scatter. Không phải chọn theo "cái nào đẹp" |
| **Một trục y duy nhất** | Biểu đồ hai trục y là lỗi phổ biến số 1 — nó cho phép tạo ra *bất kỳ* mối tương quan giả nào bằng cách chỉnh thang đo |
| **Lưới và trục phải mờ** | Chúng là nền, không phải nội dung. Dữ liệu mới là thứ nổi lên |
| **Không dùng cầu vồng cho dữ liệu có thứ tự** | Độ lớn → một màu, nhạt đến đậm. Có hai cực → hai màu + xám ở giữa |

Bảng màu dùng trong bài — **thứ tự cố định**, không xáo trộn giữa các biểu đồ:

In [ ]:
MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]   # phân loại
MAU_CHINH = "#4269d0"
MAU_NHAN = "#ff725c"      # dùng để làm NỔI một điểm cần chú ý

fig, ax = plt.subplots(figsize=(9, 1.2))
for i, m in enumerate(MAU):
    ax.add_patch(plt.Rectangle((i, 0), 0.9, 1, color=m))
ax.set_xlim(0, len(MAU)); ax.set_ylim(0, 1); ax.axis("off")
plt.title("Bảng màu phân loại — gán theo thứ tự cố định", fontsize=10)
plt.show()

> **Quan trọng:** màu phải gắn với **thực thể**, không gắn với **thứ hạng**. Nếu lọc bớt vài nhóm, các nhóm còn lại phải giữ nguyên màu cũ — nếu không người xem sẽ hiểu nhầm hoàn toàn.

## 1. Histogram — phân bố của một biến

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)

ax1.hist(df["thanh_tien"], bins=50, color=MAU_CHINH, edgecolor="white", linewidth=0.5)
ax1.set_xlabel("Giá trị đơn hàng (VND)")
ax1.set_ylabel("Số đơn")
ax1.set_title("Thang thường — bị ngoại lai kéo dẹp")

ax2.hist(df["thanh_tien"], bins=50, color=MAU_CHINH, edgecolor="white", linewidth=0.5)
ax2.set_xscale("log")
ax2.set_xlabel("Giá trị đơn hàng (thang log)")
ax2.set_title("Thang log — nhìn được hình dạng thật")

plt.tight_layout(); plt.show()

> 💡 **Mẹo dùng cả sự nghiệp:** dữ liệu tiền bạc, dân số, lượt xem hầu như luôn **lệch phải** — vài giá trị khổng lồ nén phần còn lại thành một cột dính vào trục. Đổi sang **thang log** là nhìn ra ngay hình dạng thật.

In [ ]:
# Mean vs median trên dữ liệu lệch
tb, tv = df["thanh_tien"].mean(), df["thanh_tien"].median()

plt.hist(df["thanh_tien"], bins=60, color=MAU_CHINH, edgecolor="white", linewidth=0.5)
plt.axvline(tb, color=MAU_NHAN, ls="--", lw=2, label=f"mean = {tb:,.0f}")
plt.axvline(tv, color="#3ca951", ls="--", lw=2, label=f"median = {tv:,.0f}")
plt.xlim(0, df["thanh_tien"].quantile(0.99))
plt.xlabel("Giá trị đơn hàng (VND)"); plt.ylabel("Số đơn")
plt.legend(); plt.title("Mean bị kéo lệch, median thì không")
plt.show()

print(f"Mean cao hơn median {tb/tv:.1f} lần -> phân bố lệch phải rõ rệt")

## 2. Boxplot — tìm ngoại lai

In [ ]:
danh_muc = df["danh_muc"].unique()
du_lieu = [df.loc[df["danh_muc"] == dm, "thanh_tien"] for dm in danh_muc]

fig, ax = plt.subplots(figsize=(11, 5))
bp = ax.boxplot(du_lieu, tick_labels=danh_muc, patch_artist=True, showfliers=True)

for hop, mau in zip(bp["boxes"], MAU):
    hop.set_facecolor(mau); hop.set_alpha(0.65); hop.set_edgecolor("#333")
for tp in ("whiskers", "caps", "medians"):
    for v in bp[tp]:
        v.set_color("#333")

ax.set_ylabel("Giá trị đơn hàng (VND)")
ax.set_yscale("log")
ax.set_title("Phân bố giá trị đơn theo danh mục")
plt.show()

**Đọc boxplot:**

```
        ┌─┬─────────┬─┐
   ├────┤ │    ▉    │ ├────┤        ○  ○
        └─┴─────────┴─┘
   │    │      │      │    │        │
  râu  Q1   median   Q3   râu    ngoại lai
        └── 50% dữ liệu ở giữa ──┘
```

Râu kéo dài tối đa 1.5 × IQR; các điểm ngoài đó được vẽ riêng — chính là quy tắc IQR bạn code ở bài 1.

Một biểu đồ này trả lời **bốn** câu cùng lúc: giá trị điển hình, độ phân tán, độ lệch, và có ngoại lai không.

## 3. Bar chart — so sánh giữa các nhóm

In [ ]:
dt = df.groupby("danh_muc")["thanh_tien"].sum().sort_values()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(dt.index, dt.to_numpy() / 1e9, color=MAU_CHINH, height=0.62)

for i, v in enumerate(dt.to_numpy() / 1e9):
    ax.text(v * 1.01, i, f"{v:.1f} tỷ", va="center", fontsize=9, color="#444")

ax.set_xlabel("Doanh thu (tỷ VND)")
ax.set_title("Doanh thu theo danh mục")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, dt.max() / 1e9 * 1.15)
plt.show()

**Bốn điều làm biểu đồ cột đúng:**

1. **Luôn sắp xếp** theo giá trị — trừ khi trục có thứ tự tự nhiên (tháng, bậc học vấn)
2. **Nhãn dài thì nằm ngang** (`barh`) — chữ xoay 45° rất khó đọc
3. **Ghi số trực tiếp** lên cột — người xem khỏi phải rê mắt sang trục
4. **Trục y bắt đầu từ 0** — với biểu đồ cột đây là bắt buộc, cắt trục làm phóng đại chênh lệch một cách gian dối

## 4. Line chart — xu hướng theo thời gian

In [ ]:
d = df.copy()
d["ngay"] = pd.to_datetime(d["ngay_dat"], format="%Y-%m-%d", errors="coerce")
theo_thang = d.dropna(subset=["ngay"]).set_index("ngay").resample("ME")["thanh_tien"].sum()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(theo_thang.index, theo_thang.to_numpy() / 1e9, marker="o", lw=2,
        color=MAU_CHINH, markersize=6)

dinh = theo_thang.idxmax()
ax.annotate(f"cao nhất: {theo_thang.max()/1e9:.1f} tỷ",
            xy=(dinh, theo_thang.max() / 1e9),
            xytext=(10, 12), textcoords="offset points",
            fontsize=9, color=MAU_NHAN)

ax.set_ylabel("Doanh thu (tỷ VND)")
ax.set_title("Doanh thu theo tháng")
ax.set_ylim(bottom=0)
plt.show()

> **Chỉ dùng biểu đồ đường khi trục x có thứ tự tự nhiên** (thời gian, tuổi, bậc). Nối các danh mục rời rạc bằng đường là gợi ý sai — nó ngụ ý có sự "chuyển tiếp" giữa chúng.

## 5. Nhiều nhóm — vẫn một trục y

In [ ]:
xoay = (
    d.dropna(subset=["ngay"])
     .assign(thang=lambda x: x["ngay"].dt.to_period("M").astype(str))
     .pivot_table(index="thang", columns="kenh", values="thanh_tien", aggfunc="sum")
     .fillna(0) / 1e9
)

fig, ax = plt.subplots(figsize=(11, 4.5))
for i, kenh in enumerate(xoay.columns):
    ax.plot(xoay.index, xoay[kenh], marker="o", ms=4, lw=2,
            color=MAU[i % len(MAU)], label=kenh)

ax.set_ylabel("Doanh thu (tỷ VND)")
ax.set_title("Doanh thu theo kênh")
ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.xticks(rotation=45, ha="right")
ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

> ⚠️ **Không bao giờ vẽ biểu đồ hai trục y.** Đây là lỗi phổ biến nhất trong biểu đồ kinh doanh: bằng cách chỉnh hai thang đo, bạn có thể làm *bất kỳ* hai đường nào trông như có tương quan.
>
> Hai đại lượng khác thang đo → **hai biểu đồ riêng**, hoặc quy về cùng một gốc (chỉ số hoá về 100 tại thời điểm đầu).

## 6. Scatter — quan hệ giữa hai biến

In [ ]:
mau = df.sample(600, random_state=42)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(mau["don_gia"] / 1e6, mau["so_luong"],
           alpha=0.35, s=28, color=MAU_CHINH, edgecolors="none")
ax.set_xlabel("Đơn giá (triệu VND)")
ax.set_ylabel("Số lượng")
ax.set_title(f"Đơn giá vs Số lượng   (r = {df['don_gia'].corr(df['so_luong']):.3f})")
plt.show()

> `alpha` thấp là bắt buộc khi có nhiều điểm — nếu không, các điểm chồng lên nhau thành một khối đen và bạn mất hết thông tin về mật độ.

## 7. ⚠️ Vì sao PHẢI vẽ — bộ tứ Anscombe

In [ ]:
anscombe = {
    "I":   ([10,8,13,9,11,14,6,4,12,7,5], [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]),
    "II":  ([10,8,13,9,11,14,6,4,12,7,5], [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]),
    "III": ([10,8,13,9,11,14,6,4,12,7,5], [7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]),
    "IV":  ([8,8,8,8,8,8,8,19,8,8,8],     [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]),
}

print(f"{'Bộ':<5} {'mean x':>8} {'mean y':>8} {'std x':>8} {'std y':>8} {'r':>8}")
print("-" * 50)
for ten, (x, y) in anscombe.items():
    x, y = np.array(x), np.array(y)
    print(f"{ten:<5} {x.mean():>8.2f} {y.mean():>8.2f} {x.std():>8.2f} {y.std():>8.2f} "
          f"{np.corrcoef(x, y)[0,1]:>8.3f}")

Bốn bộ dữ liệu có **mean, std và hệ số tương quan gần như y hệt nhau**. Nếu chỉ nhìn bảng này, bạn sẽ kết luận chúng giống nhau. Giờ vẽ ra:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for ax, (ten, (x, y)) in zip(axes.flat, anscombe.items()):
    x, y = np.array(x), np.array(y)
    ax.scatter(x, y, s=45, color=MAU_CHINH, zorder=3)
    he_so = np.polyfit(x, y, 1)
    xs = np.linspace(3, 20, 10)
    ax.plot(xs, np.polyval(he_so, xs), color=MAU_NHAN, lw=1.5, alpha=0.8)
    ax.set_title(f"Bộ {ten}", fontsize=11)
    ax.set_xlim(2, 20); ax.set_ylim(2, 14)

plt.tight_layout(); plt.show()

| Bộ | Thực tế |
|---|---|
| **I** | Quan hệ tuyến tính bình thường |
| **II** | Quan hệ **cong** rõ rệt — đường thẳng hoàn toàn sai |
| **III** | Tuyến tính hoàn hảo, nhưng **một ngoại lai** bẻ cong đường hồi quy |
| **IV** | Không có quan hệ nào; **một điểm duy nhất** tạo ra toàn bộ tương quan |

> 🔑 **Đây là bài học quan trọng nhất của Tuần 7.** Bốn tình huống đòi hỏi bốn cách xử lý hoàn toàn khác nhau, nhưng mọi con số thống kê đều giống hệt nhau.
>
> **Luôn vẽ. Không bao giờ chỉ nhìn số.**

## 8. Heatmap tương quan

In [ ]:
cot_so = ["so_luong", "don_gia", "giam_gia", "thanh_tien"]
tuong_quan = df[cot_so].corr()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
# Diverging: hai cực + XÁM ở giữa, luôn cố định vmin=-1, vmax=1
im = ax.imshow(tuong_quan, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(cot_so)), cot_so, rotation=45, ha="right")
ax.set_yticks(range(len(cot_so)), cot_so)
ax.grid(visible=False)

for i in range(len(cot_so)):
    for j in range(len(cot_so)):
        v = tuong_quan.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if abs(v) > 0.55 else "#222", fontsize=10)

plt.colorbar(im, ax=ax, shrink=0.8, label="hệ số tương quan")
ax.set_title("Ma trận tương quan")
plt.tight_layout(); plt.show()

**Ba quy tắc cho heatmap tương quan:**

1. **Luôn `vmin=-1, vmax=1`.** Để tự động thì thang đo đổi theo dữ liệu → không so sánh được giữa các biểu đồ, và màu đậm có thể chỉ ứng với `r = 0.2`.
2. **Dùng bảng màu phân kỳ** (hai cực + trung tính ở giữa) vì tương quan có **hai chiều** dương/âm với số 0 là mốc trung lập. Bảng cầu vồng (`jet`, `rainbow`) làm mắt thấy ranh giới giả ở những chỗ không có gì.
3. **Ghi số lên ô.** Màu cho ấn tượng nhanh, số cho độ chính xác.

> Nhớ lại bài 3 Phase 02: `r ≈ 0` **không** có nghĩa là không liên quan — nó chỉ có nghĩa là không liên quan **tuyến tính**.

## 9. Bảng tra cứu

| Câu hỏi | Dạng biểu đồ | Lưu ý |
|---|---|---|
| Biến này phân bố thế nào? | **Histogram** | Thử thang log với dữ liệu lệch |
| Có ngoại lai không? | **Boxplot** | Xem được cả 4 thông tin cùng lúc |
| Nhóm nào lớn hơn? | **Bar** | Sắp xếp; trục bắt đầu từ 0 |
| Xu hướng theo thời gian? | **Line** | Chỉ khi trục x có thứ tự |
| Hai biến có liên quan? | **Scatter** | `alpha` thấp khi nhiều điểm |
| Cột nào liên quan cột nào? | **Heatmap** | Phân kỳ, `vmin=-1, vmax=1` |
| Một con số quan trọng | **Không vẽ** — in số to | Biểu đồ 1 giá trị là thừa |

**Sáu lỗi phải tránh:**

```
❌ Hai trục y                      ❌ Biểu đồ tròn quá 5 phần
❌ Trục cột không bắt đầu từ 0     ❌ Cầu vồng cho dữ liệu có thứ tự
❌ Ghi số lên MỌI điểm             ❌ Lưới đậm hơn dữ liệu
```

## ✍️ Bài tập

Không có bài chấm tự động cho vẽ biểu đồ — bạn sẽ thực hành trực tiếp ở **project P1**. Hãy thử ngay: vẽ 3 biểu đồ bất kỳ từ `data/ban_hang.csv` và viết 1 câu nhận xét cho mỗi cái.